# 04. Wild-type Protein Structure and Mutation-Site Visualization

This notebook extends the project notebooks with a lightweight structure visualization for single-point mutations. It uses the project mutation table, downloads the available wild-type PDB structure, and highlights the amino-acid position where the mutation occurs.

No ColabFold is used here. The right panel is still the wild-type structure, but it marks the mutation site clearly so we can connect the model's ΔΔG/stability output to a concrete location on the folded protein.

## 1. Setup

This version is intentionally minimal: it only needs `pandas` and `py3Dmol`. It should run in VS Code, Jupyter, or Google Colab without GPU setup.

In [1]:
import sys
import subprocess
from pathlib import Path


def ensure_package(import_name, pip_name=None):
    pip_name = pip_name or import_name
    try:
        __import__(import_name)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pip_name])

ensure_package('pandas')
ensure_package('py3Dmol')

import os
import re
import urllib.request
from math import sqrt

import pandas as pd
import py3Dmol
from IPython.display import display

print('Python:', sys.executable)

Python: /Library/Developer/CommandLineTools/usr/bin/python3


## 2. Load the Mutation Data

This follows the same `single_point_mutations.tsv` data used by the baseline and GNN notebooks. We keep rows that have a PDB ID and a clean single-point mutation label.

In [2]:
DATA_PATH = Path('single_point_mutations.tsv')
TARGET_COL = '∆∆G_(kcal/mol)'
MUTATION_RE = re.compile(r'^\s*([A-Z])(\d+)([A-Z])')


def parse_mutation(text):
    match = MUTATION_RE.search(str(text))
    if not match:
        return pd.Series([None, None, None])
    wt, pos, mut = match.groups()
    return pd.Series([wt, int(pos), mut])


def load_examples(data_path=DATA_PATH):
    raw = pd.read_csv(data_path, sep='	')
    examples = raw.copy()
    mutation_source = examples['PDB_Chain_Mutation'].where(
        examples['PDB_Chain_Mutation'].astype(str).str.contains(MUTATION_RE, na=False),
        examples['MUTATION'],
    )
    examples[['WT_AA', 'POSITION', 'MUT_AA']] = mutation_source.apply(parse_mutation)
    examples = examples.dropna(subset=['PDB_wild', 'WT_AA', 'POSITION', 'MUT_AA', TARGET_COL]).copy()
    examples['PDB_wild'] = examples['PDB_wild'].astype(str).str.upper().str.strip()
    examples['POSITION'] = examples['POSITION'].astype(int)
    examples['mutation_label'] = examples['WT_AA'] + examples['POSITION'].astype(str) + examples['MUT_AA']
    examples['ddg'] = pd.to_numeric(examples[TARGET_COL], errors='coerce')
    examples = examples.dropna(subset=['ddg'])
    return examples.sort_values('ddg')


examples = load_examples()
print(f'Usable PDB-backed single-point mutations: {len(examples):,}')
display(examples[['PROTEIN', 'UniProt_ID', 'PDB_wild', 'mutation_label', TARGET_COL]].head(12))

Usable PDB-backed single-point mutations: 2,516


/var/folders/8n/xlm3wzj125xdpj_93jqlkhn80000gn/T/ipykernel_22998/2310450134.py:18: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  examples['PDB_Chain_Mutation'].astype(str).str.contains(MUTATION_RE, na=False),


,PROTEIN,UniProt_ID,PDB_wild,mutation_label,∆∆G_(kcal/mol)
123,Beta-glucosidase,O61594,5CG0,N391A,-23.21
273,Tailspike protein,P12528,1CLW,R383S,-17.4
829,Ribonuclease pancreatic,P61823,1RTB,Y97A,-12
830,Ribonuclease pancreatic,P61823,1RTB,Y97G,-11.7
271,Tailspike protein,P12528,1CLW,R286K,-10.3
1589,Lysozyme,P00720,2LZM,M102K,-8.9
1323,Ribonuclease,P00648,1BNI,G99V,-8.4
1322,Ribonuclease,P00648,1BNI,G100V,-7.8
151,Myelin P2 protein,P02689,6EW2,I43N,-7.63
1531,Lysozyme,P00720,2LZM,L99G,-7.4


## 3. Structure Helpers

These helpers download the wild-type PDB file, identify the best chain for the mutation, and find residues near the mutation site.

In [3]:
CACHE_DIR = Path('structure_visualization_cache')
PDB_DIR = CACHE_DIR / 'pdb_files'
PDB_DIR.mkdir(parents=True, exist_ok=True)

AA3_TO_1 = {
    'ALA':'A', 'ARG':'R', 'ASN':'N', 'ASP':'D', 'CYS':'C', 'GLN':'Q', 'GLU':'E', 'GLY':'G',
    'HIS':'H', 'ILE':'I', 'LEU':'L', 'LYS':'K', 'MET':'M', 'PHE':'F', 'PRO':'P', 'SER':'S',
    'THR':'T', 'TRP':'W', 'TYR':'Y', 'VAL':'V', 'SEC':'U', 'PYL':'O'
}


def download_pdb(pdb_id):
    pdb_id = str(pdb_id).upper().strip()
    out = PDB_DIR / f'{pdb_id}.pdb'
    if not out.exists():
        url = f'https://files.rcsb.org/download/{pdb_id}.pdb'
        print(f'Downloading wild-type PDB {pdb_id}...')
        urllib.request.urlretrieve(url, out)
    return out


def iter_ca_residues(pdb_path):
    residues = []
    seen = set()
    with open(pdb_path) as handle:
        for line in handle:
            if not line.startswith('ATOM') or line[12:16].strip() != 'CA':
                continue
            chain = line[21].strip() or 'A'
            resname = line[17:20].strip()
            resseq_text = line[22:26].strip()
            if not resseq_text or resname not in AA3_TO_1:
                continue
            key = (chain, int(resseq_text), line[26].strip())
            if key in seen:
                continue
            seen.add(key)
            residues.append({
                'chain': chain,
                'pdb_position': int(resseq_text),
                'aa': AA3_TO_1[resname],
                'x': float(line[30:38]),
                'y': float(line[38:46]),
                'z': float(line[46:54]),
            })
    return residues


def choose_chain(pdb_path, position, aa=None):
    residues = iter_ca_residues(pdb_path)
    chains = sorted({r['chain'] for r in residues})
    for chain in chains:
        exact = [r for r in residues if r['chain'] == chain and r['pdb_position'] == int(position)]
        if exact and (aa is None or exact[0]['aa'] == aa):
            return chain
    for chain in chains:
        if any(r['chain'] == chain and r['pdb_position'] == int(position) for r in residues):
            return chain
    return chains[0]


def distance(a, b):
    return sqrt((a['x'] - b['x'])**2 + (a['y'] - b['y'])**2 + (a['z'] - b['z'])**2)


def nearby_pdb_positions(pdb_path, chain, center_position, radius=6.0):
    residues = [r for r in iter_ca_residues(pdb_path) if r['chain'] == chain]
    center = next((r for r in residues if r['pdb_position'] == int(center_position)), None)
    if center is None:
        return []
    return [r['pdb_position'] for r in residues if r['pdb_position'] != int(center_position) and distance(r, center) <= radius]


def residue_at_position(pdb_path, chain, position):
    for residue in iter_ca_residues(pdb_path):
        if residue['chain'] == chain and residue['pdb_position'] == int(position):
            return residue['aa']
    return None

## 4. Visualize the Wild-type Fold and Mutation Site

The left panel highlights the wild-type amino acid in bright green. The right panel shows the same wild-type fold but marks the mutation site in red, with nearby residues shown in yellow. This gives a stable visual connection between the mutation and the folded protein location without trying to predict a new mutant fold.

In [4]:
REQUIRED_NAMES = [
    'pd', 'py3Dmol', 'load_examples', 'download_pdb', 'choose_chain',
    'iter_ca_residues', 'nearby_pdb_positions', 'residue_at_position', 'TARGET_COL'
]
missing = [name for name in REQUIRED_NAMES if name not in globals()]
if missing:
    raise RuntimeError(
        'This visualization cell needs the setup/data/helper cells above it to run first. '
        'Run all cells from the top, then rerun this one. '
        f'Missing: {missing}'
    )


def add_residue_label(view, text, selection, color, viewer, dx=34, dy=-28):
    view.addLabel(
        text,
        {
            'fontColor': color,
            'backgroundColor': 'white',
            'borderColor': color,
            'borderThickness': 1,
            'fontSize': 13,
            'showBackground': True,
            'inFront': True,
            'screenOffset': {'x': dx, 'y': dy},
        },
        selection,
        viewer=viewer,
    )


def ddg_label(value):
    direction = 'destabilizing' if value < 0 else 'stabilizing' if value > 0 else 'neutral'
    return f'ΔΔG {value:.2f} kcal/mol ({direction})'


def show_structure_pair(row):
    pdb_id = row['PDB_wild']
    mutation_label = row['mutation_label']
    wt_position = int(row['POSITION'])
    wt_aa = row['WT_AA']
    mut_aa = row['MUT_AA']

    wt_pdb = download_pdb(pdb_id)
    wt_chain = choose_chain(wt_pdb, wt_position, aa=wt_aa)
    wt_text = wt_pdb.read_text()
    observed_aa = residue_at_position(wt_pdb, wt_chain, wt_position)
    nearby_wt = nearby_pdb_positions(wt_pdb, wt_chain, wt_position, radius=6.0)
    nearby_mut = nearby_pdb_positions(wt_pdb, wt_chain, wt_position, radius=6.0)

    view = py3Dmol.view(width=1150, height=560, viewergrid=(1, 2))

    # Left panel: normal wild-type residue.
    view.addModel(wt_text, 'pdb', viewer=(0, 0))
    view.setStyle({'cartoon': {'color': 'lightgrey'}}, viewer=(0, 0))
    view.addStyle({'chain': wt_chain, 'resi': nearby_wt}, {'stick': {'color': '#ffd400', 'radius': 0.18}}, viewer=(0, 0))
    view.addStyle({'chain': wt_chain, 'resi': wt_position}, {'stick': {'colorscheme': 'limeCarbon', 'radius': 0.38}}, viewer=(0, 0))
    view.addStyle({'chain': wt_chain, 'resi': wt_position}, {'sphere': {'color': '#00ff00', 'radius': 1.1, 'opacity': 0.9}}, viewer=(0, 0))
    add_residue_label(view, f'WT {wt_aa}{wt_position}', {'chain': wt_chain, 'resi': wt_position}, '#00cc00', (0, 0))

    # Right panel: same fold, mutation site emphasized.
    view.addModel(wt_text, 'pdb', viewer=(0, 1))
    view.setStyle({'cartoon': {'color': 'lightgrey'}}, viewer=(0, 1))
    view.addStyle({'chain': wt_chain, 'resi': nearby_mut}, {'stick': {'color': '#ffd400', 'radius': 0.18}}, viewer=(0, 1))
    view.addStyle({'chain': wt_chain, 'resi': wt_position}, {'stick': {'colorscheme': 'redCarbon', 'radius': 0.42}}, viewer=(0, 1))
    view.addStyle({'chain': wt_chain, 'resi': wt_position}, {'sphere': {'color': '#ff0000', 'radius': 1.15, 'opacity': 0.9}}, viewer=(0, 1))
    add_residue_label(view, f'Mutation {wt_aa}{wt_position}{mut_aa}', {'chain': wt_chain, 'resi': wt_position}, '#cc0000', (0, 1))

    focus = {'chain': wt_chain, 'resi': wt_position}
    view.zoomTo(focus, viewer=(0, 0))
    view.zoomTo(focus, viewer=(0, 1))

    print(
        f'PDB={pdb_id} | chain={wt_chain} | mutation={mutation_label} | '
        f'PDB residue={observed_aa}{wt_position} | nearby residues highlighted={len(nearby_mut)}'
    )
    return view.show()


if 'examples' not in globals():
    examples = load_examples()

example_index = 0
selected = examples.iloc[example_index]
print(selected[['PROTEIN', 'PDB_wild', 'mutation_label', TARGET_COL]])
show_structure_pair(selected)

PROTEIN           Beta-glucosidase
PDB_wild                      5CG0
mutation_label               N391A
∆∆G_(kcal/mol)              -23.21
Name: 123, dtype: object
PDB=5CG0 | chain=A | mutation=N391A | PDB residue=N391 | nearby residues highlighted=3


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## 5. Known Experimental Mutant Structures

The dataset does **not** include a `PDB_mutant` column, so most rows only point to the wild-type/reference PDB. However, some mutations have separate experimentally solved mutant PDB structures in RCSB.

This section starts a small manually curated list. The first example is T4 lysozyme `R96H`: the table has wild-type/reference PDB `2LZM`, and RCSB has related high-resolution structures from the Arg96→His mutant study, including `3F8V`.

In [7]:
KNOWN_MUTANT_STRUCTURE_PAIRS = pd.DataFrame([
    {
        'protein': 'Lysozyme',
        'uniprot_id': 'P00720',
        'mutation_label': 'R96H',
        'wt_pdb': '2LZM',
        'mutant_pdb': '3F8V',
        'mutation_position': 96,
        'wt_aa': 'R',
        'mut_aa': 'H',
        'note': 'Experimental T4 lysozyme R96H mutant structure.',
    },
    {
        'protein': 'Lysozyme',
        'uniprot_id': 'P00720',
        'mutation_label': 'M102K',
        'wt_pdb': '2LZM',
        'mutant_pdb': '1L54',
        'mutation_position': 102,
        'wt_aa': 'M',
        'mut_aa': 'K',
        'note': 'Experimental T4 lysozyme M102K mutant structure.',
    },
    {
        'protein': 'RNase Sa',
        'uniprot_id': 'P05798',
        'mutation_label': 'S24A',
        'wt_pdb': '1RGG',
        'mutant_pdb': '4GHO',
        'mutation_position': 24,
        'wt_aa': 'S',
        'mut_aa': 'A',
        'note': 'Experimental RNase Sa S24A mutant structure.',
    },
    {
        'protein': 'RNase Sa',
        'uniprot_id': 'P05798',
        'mutation_label': 'T95A',
        'wt_pdb': '1RGG',
        'mutant_pdb': '4J5G',
        'mutation_position': 95,
        'wt_aa': 'T',
        'mut_aa': 'A',
        'note': 'Experimental RNase Sa T95A mutant structure.',
    },
    {
        'protein': 'RNase Sa',
        'uniprot_id': 'P05798',
        'mutation_label': 'Y51F',
        'wt_pdb': '1RGG',
        'mutant_pdb': '4J5K',
        'mutation_position': 51,
        'wt_aa': 'Y',
        'mut_aa': 'F',
        'note': 'Experimental RNase Sa Y51F mutant structure.',
    },
    {
        'protein': 'Superoxide dismutase [Mn]',
        'uniprot_id': 'P04179',
        'mutation_label': 'Y34F',
        'wt_pdb': '1N0J',
        'mutant_pdb': '4E4E',
        'mutation_position': 34,
        'wt_aa': 'Y',
        'mut_aa': 'F',
        'note': 'Experimental yeast manganese superoxide dismutase Y34F mutant structure.',
    },
    {
        'protein': 'Myelin P2 protein',
        'uniprot_id': 'P02689',
        'mutation_label': 'I43N',
        'wt_pdb': '6EW2',
        'mutant_pdb': '5N4M',
        'mutation_position': 43,
        'wt_aa': 'I',
        'mut_aa': 'N',
        'note': 'Experimental human myelin P2 I43N mutant structure.',
    },
    {
        'protein': 'Thiol:disulfide interchange protein DsbA',
        'uniprot_id': 'P0AEG4',
        'mutation_label': 'H32Y',
        'wt_pdb': '1A23',
        'mutant_pdb': '1FVJ',
        'mutation_position': 32,
        'wt_aa': 'H',
        'mut_aa': 'Y',
        'note': 'Experimental DsbA H32Y mutant structure.',
    },
    {
        'protein': 'Ribonuclease T1',
        'uniprot_id': 'P00651',
        'mutation_label': 'V16S',
        'wt_pdb': '1RN1',
        'mutant_pdb': '1G02',
        'mutation_position': 16,
        'wt_aa': 'V',
        'mut_aa': 'S',
        'note': 'Experimental RNase T1 V16S mutant structure.',
    },
    {
        'protein': 'Trypsin inhibitor',
        'uniprot_id': 'P00974',
        'mutation_label': 'F45A',
        'wt_pdb': '1BPI',
        'mutant_pdb': '1NAG',
        'mutation_position': 45,
        'wt_aa': 'F',
        'mut_aa': 'A',
        'note': 'BPTI mutant-structure entry from F22A/Y23A/N43G/F45A crystal structure series.',
    },
    {
        'protein': 'Cytochrome b5',
        'uniprot_id': 'P00171',
        'mutation_label': 'V45E',
        'wt_pdb': '1CYO',
        'mutant_pdb': '1LQX',
        'mutation_position': 45,
        'wt_aa': 'V',
        'mut_aa': 'E',
        'note': 'Experimental cytochrome b5 V45E mutant structure.',
    }
])

display(KNOWN_MUTANT_STRUCTURE_PAIRS)


def find_matching_dataset_row(pair):
    matches = examples[
        (examples['PDB_wild'].astype(str).str.upper() == pair['wt_pdb'])
        & (examples['mutation_label'] == pair['mutation_label'])
    ]
    if len(matches):
        return matches.iloc[0]
    return None


def show_experimental_mutant_pair(pair_index=0):
    pair = KNOWN_MUTANT_STRUCTURE_PAIRS.iloc[pair_index]
    dataset_row = find_matching_dataset_row(pair)

    wt_pdb = download_pdb(pair['wt_pdb'])
    mutant_pdb = download_pdb(pair['mutant_pdb'])
    position = int(pair['mutation_position'])

    wt_chain = choose_chain(wt_pdb, position, aa=pair['wt_aa'])
    mutant_chain = choose_chain(mutant_pdb, position, aa=pair['mut_aa'])
    wt_text = wt_pdb.read_text()
    mutant_text = mutant_pdb.read_text()

    wt_observed = residue_at_position(wt_pdb, wt_chain, position)
    mutant_observed = residue_at_position(mutant_pdb, mutant_chain, position)
    wt_nearby = nearby_pdb_positions(wt_pdb, wt_chain, position, radius=6.0)
    mutant_nearby = nearby_pdb_positions(mutant_pdb, mutant_chain, position, radius=6.0)

    view = py3Dmol.view(width=1150, height=560, viewergrid=(1, 2))

    view.addModel(wt_text, 'pdb', viewer=(0, 0))
    view.setStyle({'cartoon': {'color': 'lightgrey'}}, viewer=(0, 0))
    view.addStyle({'chain': wt_chain, 'resi': wt_nearby}, {'stick': {'color': '#ffd400', 'radius': 0.18}}, viewer=(0, 0))
    view.addStyle({'chain': wt_chain, 'resi': position}, {'stick': {'colorscheme': 'limeCarbon', 'radius': 0.42}}, viewer=(0, 0))
    view.addStyle({'chain': wt_chain, 'resi': position}, {'sphere': {'color': '#00ff00', 'radius': 1.15, 'opacity': 0.9}}, viewer=(0, 0))
    add_residue_label(view, f"WT {pair['wt_aa']}{position}", {'chain': wt_chain, 'resi': position}, '#00cc00', (0, 0))

    view.addModel(mutant_text, 'pdb', viewer=(0, 1))
    view.setStyle({'cartoon': {'color': 'lightgrey'}}, viewer=(0, 1))
    view.addStyle({'chain': mutant_chain, 'resi': mutant_nearby}, {'stick': {'color': '#ffd400', 'radius': 0.18}}, viewer=(0, 1))
    view.addStyle({'chain': mutant_chain, 'resi': position}, {'stick': {'colorscheme': 'redCarbon', 'radius': 0.42}}, viewer=(0, 1))
    view.addStyle({'chain': mutant_chain, 'resi': position}, {'sphere': {'color': '#ff0000', 'radius': 1.15, 'opacity': 0.9}}, viewer=(0, 1))
    add_residue_label(view, f"Mutation {pair['wt_aa']}{position}{pair['mut_aa']}", {'chain': mutant_chain, 'resi': position}, '#cc0000', (0, 1))

    wt_title = f"WT/reference {pair['wt_pdb']} chain {wt_chain}"
    mutant_title = f"Experimental mutant {pair['mutant_pdb']} | {pair['mutation_label']}"
    if dataset_row is not None:
        mutant_title += f" | ΔΔG {dataset_row['ddg']:.2f} kcal/mol"

    view.zoomTo({'chain': wt_chain, 'resi': position}, viewer=(0, 0))
    view.zoomTo({'chain': mutant_chain, 'resi': position}, viewer=(0, 1))

    print(
        f"WT observed residue: {wt_observed}{position} in {pair['wt_pdb']} chain {wt_chain} | "
        f"Mutant observed residue: {mutant_observed}{position} in {pair['mutant_pdb']} chain {mutant_chain}"
    )
    if dataset_row is not None:
        print(dataset_row[['PROTEIN', 'PDB_wild', 'mutation_label', TARGET_COL]])
    else:
        print('No exact matching row found in examples; showing curated PDB pair only.')

    return view.show()


show_experimental_mutant_pair(10)

,protein,uniprot_id,mutation_label,wt_pdb,mutant_pdb,mutation_position,wt_aa,mut_aa,note
0,Lysozyme,P00720,R96H,2LZM,3F8V,96,R,H,Experimental T4 lysozyme R96H mutant structure.
1,Lysozyme,P00720,M102K,2LZM,1L54,102,M,K,Experimental T4 lysozyme M102K mutant structure.
2,RNase Sa,P05798,S24A,1RGG,4GHO,24,S,A,Experimental RNase Sa S24A mutant structure.
3,RNase Sa,P05798,T95A,1RGG,4J5G,95,T,A,Experimental RNase Sa T95A mutant structure.
4,RNase Sa,P05798,Y51F,1RGG,4J5K,51,Y,F,Experimental RNase Sa Y51F mutant structure.
5,Superoxide dismutase [Mn],P04179,Y34F,1N0J,4E4E,34,Y,F,Experimental yeast manganese superoxide dismut...
6,Myelin P2 protein,P02689,I43N,6EW2,5N4M,43,I,N,Experimental human myelin P2 I43N mutant struc...
7,Thiol:disulfide interchange protein DsbA,P0AEG4,H32Y,1A23,1FVJ,32,H,Y,Experimental DsbA H32Y mutant structure.
8,Ribonuclease T1,P00651,V16S,1RN1,1G02,16,V,S,Experimental RNase T1 V16S mutant structure.
9,Trypsin inhibitor,P00974,F45A,1BPI,1NAG,45,F,A,BPTI mutant-structure entry from F22A/Y23A/N43...


WT observed residue: V45 in 1CYO chain A | Mutant observed residue: E45 in 1LQX chain A
PROTEIN           Cytochrome b5
PDB_wild                   1CYO
mutation_label             V45E
∆∆G_(kcal/mol)            -2.68
Name: 512, dtype: object


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## 6. Try Another Mutation

Change `example_index` to inspect another row from `examples`. Because this version only downloads PDB files, repeat runs should be fast after the first download.

In [6]:
# Example: uncomment and change the number to inspect another mutation.
# example_index = 5
# selected = examples.iloc[example_index]
# print(selected[['PROTEIN', 'PDB_wild', 'mutation_label', TARGET_COL]])
# show_structure_pair(selected)